# 01 — Analyse Exploratoire (EDA)

**Projet : ObRail — Détection des sous-dessertes ferroviaires**

**Objectif** : explorer le jeu de données enrichi pour comprendre sa structure, valider sa qualité, et **vérifier si les features permettent de prédire la cible `is_underserved`** (ligne sous-desservie : demande > offre) *avant* la phase de modélisation.

- **Input** : `../data/processed/routes_processed.csv`
- **Outputs** : observations documentées + figures dans `../evaluation/plots/`
- **Type de problème** : classification binaire (`is_underserved` ∈ {0,1}), classes déséquilibrées
- **Date** : 2026

## §1 — Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

PLOTS_DIR = Path('../evaluation/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup OK')

## §2 — Chargement des données

In [ ]:
df = pd.read_csv('../data/processed/routes_processed.csv')
print('Dimensions :', df.shape)

## §3 — Aperçu général

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## §4 — Qualité des données

In [ ]:
print('Valeurs manquantes par colonne (>0) :')
na = df.isnull().sum()
print(na[na > 0] if (na > 0).any() else 'Aucune valeur manquante')
print('\nLignes dupliquées :', df.duplicated().sum())

In [ ]:
# Bornes des features clés (détection de valeurs aberrantes)
for c in ['distance_km','load_factor','capacity','passengers_estimated','service_ratio']:
    print(f'{c:22s} min={df[c].min():.3f}  max={df[c].max():.3f}')

# Features à variance nulle (inutiles pour le modèle)
const_cols = [c for c in df.select_dtypes('number').columns if df[c].nunique() == 1]
print('\nFeatures constantes (variance nulle) :', const_cols)

**Déduction §4** : données **propres** (0 valeur manquante, 0 doublon).
Plusieurs colonnes liées aux émissions sont **constantes ou quasi-constantes** (`train_gco2_pkm`=14, `plane_gco2_pkm`=144, `savings_percent`≈90 %, `co2_ratio_train_plane`≈0,097) — vestiges de l'ancien sujet « CO₂ », **sans valeur prédictive** pour la sous-desserte.

## §5 — Analyse de la cible `is_underserved` ⭐

In [ ]:
counts = df['is_underserved'].value_counts()
pct = (df['is_underserved'].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({'effectif': counts, 'part_%': pct}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False, ax=ax, palette=['#4c72b0', '#dd8452'])
ax.set_title("Distribution de la cible is_underserved")
ax.set_xlabel('is_underserved'); ax.set_ylabel('Nombre de lignes')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f'{v}\n({pct.iloc[i]}%)', ha='center')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'eda_target_balance.png', dpi=110); plt.show()

**Déduction §5** : forte **asymétrie de classes** — ≈ **89,6 % de `0`** contre **10,4 % de `1`**.
Conséquences pour l'évaluation :
- l'**accuracy est trompeuse** : un modèle prédisant toujours `0` atteint déjà ~89,6 % ;
- privilégier **precision / recall / F1** sur la classe positive, **ROC-AUC** et surtout **PR-AUC** ;
- prévoir un **rééquilibrage** (oversampling/SMOTE) à l'entraînement.

## §6 — Analyse univariée des features

In [ ]:
# Variables catégorielles
for c in ['service_type','type','distance_category','is_cross_border']:
    print(f'-- {c} --'); print(df[c].value_counts(), '\n')
print('-- top 8 opérateurs --'); print(df['operator'].value_counts().head(8))
print('\nPays origine / destination distincts :', df['origin_country'].nunique(), '/', df['destination_country'].nunique())
print('service_type identique à type ?', (df['service_type'] == df['type']).all())

In [ ]:
# Distributions des features numériques
num_feats = ['distance_km','load_factor','capacity','passengers_estimated','service_ratio']
fig, axes = plt.subplots(1, len(num_feats), figsize=(20, 4))
for ax, c in zip(axes, num_feats):
    sns.histplot(df[c], kde=True, ax=ax, color='#4c72b0')
    ax.set_title(c)
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'eda_distributions.png', dpi=110); plt.show()

**Déduction §6** : `service_type` et `type` sont **strictement identiques** (colonne dupliquée → en supprimer une).
`distance_km` est très asymétrique (longue traîne) ; `capacity` est quasi-binaire (≈400/500) ; `is_cross_border` est rare (79 lignes).

## §7 — Features vs cible `is_underserved` ⭐

In [ ]:
num = ['distance_km','load_factor','capacity','passengers_estimated','service_ratio','is_cross_border']
print('Moyenne des features par classe :')
print(df.groupby('is_underserved')[num].mean().T)
print('\nCorrélation des features avec la cible :')
print(df[num + ['is_underserved']].corr()['is_underserved'].drop('is_underserved').sort_values())

In [ ]:
# Boxplots des features numériques selon la classe
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, c in zip(axes, ['distance_km','load_factor','service_ratio','passengers_estimated']):
    sns.boxplot(data=df, x='is_underserved', y=c, hue='is_underserved', legend=False, ax=ax, palette=['#4c72b0', '#dd8452'])
    ax.set_title(f'{c} par classe')
plt.tight_layout(); plt.savefig(PLOTS_DIR / 'eda_features_vs_target.png', dpi=110); plt.show()

In [ ]:
# Test de fuite de cible : is_underserved est-il un simple seuil sur service_ratio / load_factor ?
for c in ['service_ratio','load_factor']:
    g0 = df.loc[df.is_underserved == 0, c]; g1 = df.loc[df.is_underserved == 1, c]
    chevauche = g1.max() > g0.min() and g0.max() > g1.min()
    print(f'{c}: classe0 [{g0.min():.3f}, {g0.max():.3f}]  classe1 [{g1.min():.3f}, {g1.max():.3f}]  -> chevauchement: {chevauche}')

In [ ]:
# Taux de sous-desserte par variable catégorielle / géographique
print('Par distance_category :'); print((df.groupby('distance_category')['is_underserved'].mean()*100).round(1))
top = df['origin_country'].value_counts().head(10).index
print('\nPar pays origine (top 10 volume) :'); print((df[df.origin_country.isin(top)].groupby('origin_country')['is_underserved'].mean()*100).round(1).sort_values(ascending=False))
topop = df['operator'].value_counts().head(10).index
print('\nPar opérateur (top 10 volume) :'); print((df[df.operator.isin(topop)].groupby('operator')['is_underserved'].mean()*100).round(1).sort_values(ascending=False))

**Déduction §7 (centrale)** :
- 🔴 Les features **numériques** (`load_factor`, `capacity`, `passengers_estimated`, `service_ratio`) ne séparent **pas** les classes : corrélations ≈ 0 et moyennes quasi identiques entre `0` et `1`. Or ce sont les features utilisées par l'API de prédiction.
- ✅ **Pas de fuite de cible** : les plages de `service_ratio`/`load_factor` se chevauchent entre classes (la cible n'est pas un seuil déterministe).
- 🟡 **Signal faible mais réel** dans les variables **catégorielles/géographiques** : `distance_category` (très long ↑), `origin_country` (BE ≈ 26 %), `operator` (4,8 %→13,3 %).

## §8 — Couverture géographique

In [ ]:
rate_country = (df.groupby('origin_country')['is_underserved'].mean()*100)
vol_country = df['origin_country'].value_counts()
geo = pd.DataFrame({'volume': vol_country, 'taux_sous_desserte_%': rate_country.round(1)}).sort_values('volume', ascending=False).head(12)
print(geo)

fig, ax = plt.subplots(figsize=(11, 4))
geo_sorted = geo.sort_values('taux_sous_desserte_%', ascending=False)
sns.barplot(x=geo_sorted.index, y=geo_sorted['taux_sous_desserte_%'], ax=ax, color='#dd8452')
ax.axhline(df.is_underserved.mean()*100, ls='--', color='grey', label='taux global')
ax.set_title('Taux de sous-desserte par pays (top 12 volume)'); ax.set_ylabel('% sous-desservi'); ax.legend()
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(PLOTS_DIR / 'eda_underserved_by_country.png', dpi=110); plt.show()

**Déduction §8** : le dataset est concentré sur l'Europe de l'Ouest (FR, DE, CH dominent). Les taux de sous-desserte varient par pays, mais certains pays ont de très faibles effectifs → estimations peu robustes.

## §9 — Synthèse & implications pour la modélisation

**Qualité** : données propres (0 manquant, 0 doublon), 3 678 lignes, 29 colonnes.

**Nettoyage recommandé** (pour `02_feature_engineering`) :
- supprimer les colonnes **constantes** (`train_gco2_pkm`, `plane_gco2_pkm`, etc.) et les colonnes **dupliquées** (`type` == `service_type`) ;
- les variables liées au CO₂ relèvent de l'**ancien sujet** → à écarter de la modélisation sous-desserte.

**Cible** : `is_underserved`, **binaire et déséquilibrée** (~10,4 % positifs).

**Point d'alerte majeur** : les features numériques cœur **n'expliquent pas la cible** (corrélations ≈ 0). Le signal exploitable est **faible** et surtout **catégoriel/géographique** (pays, opérateur, catégorie de distance).

**Conséquences** :
1. attentes réalistes : il sera **difficile de battre le baseline de 89,6 %** d'accuracy ;
2. **évaluation** : juger sur **recall/precision/F1 (classe 1)**, **ROC-AUC** et **PR-AUC**, pas l'accuracy ;
3. **modélisation** : encoder soigneusement les catégorielles, rééquilibrer les classes ;
4. **recommandation amont** : réexaminer la **définition / construction de `is_underserved`** et envisager d'enrichir les features (la sous-desserte « demande > offre » devrait logiquement dépendre de `load_factor`/`service_ratio`, ce qui n'est pas le cas ici).